In [ ]:
import re

import pandas as pd

# 1. Clinical Features

In [ ]:
# read in data
clinical_features = pd.read_excel("./data/matched_clinical_features.xlsx")


# infer Event 
clinical_features["event"] = clinical_features.apply(
    lambda row: True if row["overall_survival"] > row["event_free_survival"] else False,
    axis=1,
)


# select and rename columns
clinical_features = clinical_features[
    [
        "SubjectID",
         "legal_sex",
        "age_at_event_days",
        "consolidated_tumor_locations",
        "cancer_predisposition",
        "extent_of_tumor_resection",
        "chemotherapy",
        "radiation",
        "event_free_survival",
        "event",
        "Cohort",
    ]
]
clinical_features = clinical_features.rename(
    columns={
        "legal_sex": "Sex",
        "age_at_event_days": "Age at Diagnosis",
        "consolidated_tumor_locations": "Tumor Location",
        "cancer_predisposition": "NF1",
        "extent_of_tumor_resection": "Extent of Tumor Resection",
        "chemotherapy": "Chemotherapy",
        "radiation": "Radiation",
        "event_free_survival": "Progression Free Survival",
        "event": "Event",
    }
)

# set index
clinical_features.set_index("SubjectID", inplace=True)

# cache clinical features
clinical_features.to_pickle("./data/clinical_features.pkl")

print(clinical_features.shape)
print(clinical_features.columns)


# 2. Radiomic Features

In [ ]:
# read in data
Resnet_features = pd.read_csv("./data/plgg_features_layer3_gap_gmp_20260308_232319.csv")

# extract subject IDs
Resnet_features["SubjectID"] = Resnet_features["SubjectID"].apply(
    lambda x: (match := re.match(r"^(C\d+|sub\d+)", x)) and match.group()
)

# filter subjects
Resnet_features = Resnet_features[
    Resnet_features["SubjectID"].isin(clinical_features.index)
]

# set index
Resnet_features.set_index("SubjectID", inplace=True)


# cache radiomic features 
Resnet_features.to_pickle("./data/Resnet_features.pkl")

print("Final ResNet feature matrix shape:", Resnet_features.shape)


In [ ]:
Resnet_features